In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
weather_df = pd.read_pickle('full_weather.pkl')
hurdat_df = pd.read_pickle("hurdat2_storm_data_2020_2025.pkl")

# Ensure timestamps are datetime
weather_df['date'] = pd.to_datetime(weather_df['date'])
hurdat_df['storm_timestamp'] = pd.to_datetime(hurdat_df['datetime'])

journeys_storms = []

for journey_id, group in tqdm(weather_df.groupby('journey_id'), total=weather_df['journey_id'].nunique(), desc="Collecting storms for journeys"):
    start_time = group['date'].min()
    end_time = group['date'].max()
    # Filter storms within journey timeframe and above wind threshold
    storms_in_journey = hurdat_df[
        (hurdat_df['storm_timestamp'] >= start_time) &
        (hurdat_df['storm_timestamp'] <= end_time) &
        (pd.to_numeric(hurdat_df['max_sustained_wind'], errors='coerce') > 30)
    ]
    for _, storm in storms_in_journey.iterrows():
        journeys_storms.append({
            'journey_id': journey_id,
            'storm_id': storm['storm_id'],
            'storm_name': storm['storm_name'],
            'system_status': storm['system_status'],
            'lat': storm['lat'],
            'lon': storm['lon'],
            'max_sustained_wind': storm['max_sustained_wind'],
            'datetime': storm['storm_timestamp'],
            'system_status_desc': storm['system_status_desc']
        })

journeys_storms_df = pd.DataFrame(journeys_storms)
journeys_storms_df.to_pickle('journeys_storms.pkl')
journeys_storms_df.to_csv('journeys_storms.csv', index=False)